# Irina: SWE-bench + local Qwen + GRPO

**Research implementation:** the original Odysseus Python source was not supplied, so this is an independent harness following the documented design. Docker-backed model execution, official verification and hosted Colab execution remain unverified. See `VALIDATION.md`.


In [ ]:
# Configuration — educational is the default
MODE = "educational"
MODEL_ID = "Qwen/Qwen2.5-Coder-0.5B-Instruct"
GROUP_SIZE = 4
MAX_AGENT_TURNS = 4
SEED = 42
TASK_INDEX = 0
REPO_URL = ""  # Set to your published Irina GitHub clone URL


In [ ]:
# Bootstrap from a real GitHub repository; publishing this ZIP is a separate step.
import os, subprocess, sys
if not REPO_URL:
    raise RuntimeError("Set REPO_URL to your published Irina GitHub repository URL")
subprocess.run(["git", "clone", "--depth", "1", REPO_URL, "/content/Irina"], check=True)
subprocess.run([sys.executable, "-m", "pip", "install", "uv"], check=True)
subprocess.run([sys.executable, "-m", "uv", "sync", "--extra", "train", "--project", "/content/Irina"], check=True)


## Chapter 0 — The big picture


```mermaid
flowchart TD
    A["① Submit SWE-bench task"] --> B["Load task metadata"]
    B --> C["Extract repository, base_commit, issue text"]
    C --> D["② Prepare isolated execution workspace"]
    D --> E["Clone or locate repository"]
    E --> F["Checkout exact base_commit"]
    F --> G["③ Initialize Irina harness"]

    G --> H["Build system prompt and task context"]
    H --> I["Send conversation to local Qwen policy"]
    I --> J["Qwen generates response"]
    J --> K{"Tool call requested?"}

    K -->|Yes| L["Irina parses tool name and arguments"]
    L --> M["Apply tool security policy"]
    M --> N{"Tool permitted?"}
    N -->|No| O["Return BLOCKED observation"]
    N -->|Yes| P{"Requested tool"}

    P -->|read_file| Q["Read real repository file"]
    P -->|grep / list_files| R["Search real repository"]
    P -->|edit_file / write_file| S["Modify repository files"]
    P -->|bash| T["Run command in isolated environment"]

    Q --> U["Capture tool output"]
    R --> U
    S --> U
    T --> U
    O --> V["Append observation to conversation"]
    U --> V

    V --> W{"Agent finished or turn limit reached?"}
    W -->|No| I
    W -->|Yes| X["④ Finalize trajectory"]
    K -->|No| X

    X --> Y["Collect git diff"]
    Y --> Z{"Patch valid and applicable?"}

    Z -->|No| AA["Assign invalid-patch reward"]
    Z -->|Yes| MODE{"Reward mode?"}

    MODE -->|Educational| RAND["Generate synthetic random reward"]
    MODE -->|Realistic| AB["⑤ Send patch to independent verifier"]

    AB --> AC["Create clean verification workspace"]
    AC --> AD["Checkout base_commit"]
    AD --> AE["Apply candidate patch"]
    AE --> AF["Apply SWE-bench test patch"]
    AF --> AG["Run FAIL_TO_PASS and PASS_TO_PASS tests"]
    AG --> AH["Collect test results"]
    AH --> AI["Compute correctness reward"]

    AA --> AJ["Store trajectory and reward"]
    RAND --> AJ
    AI --> AJ

    AJ --> AK{"⑥ Remaining sampled rollouts?"}
    AK -->|Yes| D
    AK -->|No| AL["Compute group-relative advantages"]
    AL --> AM["Compute policy loss on generated tokens"]
    AM --> AN["Backpropagate through Qwen LoRA adapters"]
    AN --> AO["Optimizer step"]
    AO --> AP["⑦ Updated Qwen policy"]

    classDef prep fill:#DBEAFE,stroke:#2563EB,color:#1E3A8A,stroke-width:2px
    classDef agent fill:#EDE9FE,stroke:#7C3AED,color:#4C1D95,stroke-width:2px
    classDef tools fill:#DCFCE7,stroke:#16A34A,color:#14532D,stroke-width:2px
    classDef verify fill:#FFEDD5,stroke:#EA580C,color:#7C2D12,stroke-width:2px
    classDef learn fill:#E5E7EB,stroke:#64748B,color:#1E293B,stroke-width:2px

    class A,B,C,D,E,F prep
    class G,H,I,J,K,L,M,N,O,V,W,X agent
    class P,Q,R,S,T,U tools
    class Y,Z,AA,MODE,AB,AC,AD,AE,AF,AG,AH,AI verify
    class RAND tools
    class AJ,AK,AL,AM,AN,AO,AP learn

    linkStyle default stroke:#94A3B8,stroke-width:1.5px
```


## Chapter 1 — Meet the SWE-bench task


In [ ]:
from irina.rl.tasks import load_task
task = load_task(TASK_INDEX)
print(task.instance_id, task.repo, task.base_commit)
print(task.problem_statement[:1000])


## Chapter 2 — Prepare the repository


## Chapter 3 — Meet Irina


In [ ]:
from irina import Harness, Policy
print(Harness, Policy)


## Chapter 4 — Connect local Qwen


## Chapter 5 — One agent rollout


```mermaid
flowchart TD
    A["① SWE-bench problem statement"] --> B["Irina builds conversation"]
    B --> C["② Qwen generates response"]
    C --> D{"Response type?"}

    D -->|Tool call| E["③ Irina parses tool name and arguments"]
    E --> F["Apply tool security policy"]
    F --> G{"Tool permitted?"}

    G -->|No| H["Return BLOCKED observation"]
    G -->|Yes| I{"Requested tool?"}

    I -->|read_file| J["Read real repository file"]
    I -->|grep / list_files| K["Search real repository"]
    I -->|edit_file / write_file| L["Modify real repository files"]
    I -->|bash| M["Execute command in isolated environment"]

    J --> N["④ Capture tool output"]
    K --> N
    L --> N
    M --> N

    H --> O["Append observation to conversation"]
    N --> O

    O --> P{"Agent finished or turn limit reached?"}
    P -->|No| C
    P -->|Yes| Q["⑤ Finalize trajectory"]

    D -->|Final answer| Q

    Q --> R["Save messages, generated tokens and logprobs"]
    R --> S["⑥ Collect candidate patch using git diff"]
```


## Chapter 6 — Collect the patch


## Chapter 7 — Two reward paths


```mermaid
flowchart TD
    A["① Agent rollout completed"] --> B["Collect candidate git diff"]
    B --> C{"② Patch valid and applicable?"}

    C -->|No| D["Assign invalid-patch reward"]
    C -->|Yes| E{"③ Reward mode?"}

    E -->|Educational| F["Synthetic reward backend"]
    F --> G["Generate seeded random reward"]
    G --> H["Return synthetic reward"]

    E -->|Realistic| I["④ Independent SWE-bench verifier"]
    I --> J["Create clean verification workspace"]
    J --> K["Checkout exact base_commit"]
    K --> L["Apply candidate patch"]
    L --> M["Apply SWE-bench test patch"]
    M --> N["⑤ Run FAIL_TO_PASS and PASS_TO_PASS tests"]
    N --> O["Collect test results"]
    O --> P{"⑥ Verification result?"}

    P -->|Resolved| Q["Compute success reward"]
    P -->|Unresolved| R["Compute failure reward"]
    P -->|Infrastructure error| S["Record evaluation error separately"]

    D --> T["⑦ Store trajectory and reward"]
    H --> T
    Q --> T
    R --> T
    S --> U["Retry or exclude according to evaluation policy"]
    U --> T
```


In [ ]:
from irina.rl.rewards import synthetic_reward
print("Reward mode:", MODE)
print("Example seeded synthetic reward:", synthetic_reward("diff --git a/a b/a", SEED, 0))


## Chapter 8 — Grouped rollouts


```mermaid
flowchart TD
    A["① One SWE-bench task"] --> B["Freeze starting Qwen policy"]

    B --> C["② Rollout 1: fresh workspace"]
    B --> D["② Rollout 2: fresh workspace"]
    B --> E["② Rollout 3: fresh workspace"]
    B --> F["② Rollout N: fresh workspace"]

    C --> G["Trajectory 1 + candidate patch 1"]
    D --> H["Trajectory 2 + candidate patch 2"]
    E --> I["Trajectory 3 + candidate patch 3"]
    F --> J["Trajectory N + candidate patch N"]

    G --> K["③ Selected reward backend"]
    H --> K
    I --> K
    J --> K

    K --> L{"Reward mode?"}
    L -->|Educational| M["Synthetic random rewards"]
    L -->|Realistic| N["Independent SWE-bench verification"]

    M --> O["④ Collect rewards r1, r2, ..., rN"]
    N --> O

    O --> P["Compute group mean and standard deviation"]
    P --> Q["⑤ Calculate group-relative advantages"]
    Q --> R["Associate each advantage with generated tokens"]
    R --> S["⑥ Compute policy-gradient loss"]
    S --> T["Backpropagate through Qwen LoRA adapters"]
    T --> U["⑦ Optimizer step"]
    U --> V["Updated Qwen policy"]
```


## Chapter 9 — Advantages


In [ ]:
from irina.rl.training import advantages
print(advantages([0.2, 0.8, 0.5, 0.1]))


## Chapter 10 — Generated tokens and loss


## Chapter 11 — One LoRA step


In [ ]:
# Requires a secure writable sandbox before attempting an end-to-end run.
from irina.rl import run_pipeline
from irina.rl.config import Config
print("Requested configuration:", Config(mode=MODE, model_id=MODEL_ID, group_size=GROUP_SIZE, max_turns=MAX_AGENT_TURNS, seed=SEED, task_index=TASK_INDEX))
print("Do not run until secure execution backend is configured; see README.")


## Chapter 12 — Results and limitations


The synthetic reward is not a correctness metric. Official SWE-bench verification, original Odysseus features, and secure writable tool execution remain future integration work.
